In [ ]:
"""
Task 8: Multi-Agent Collaborative Swarm with Shared Blackboard
Simplified, runnable version: a real deployment swaps SharedBlackboard's
in-memory dict + threading.Lock for Redis (redis-py `SETNX` locks) and the
`commit()` call for a Postgres INSERT, and each Agent's `.act()` for an
AutoGen / OpenAI-API call. The coordination logic (locking, negotiation,
deadlock avoidance, commit) is identical either way.
"""

import threading
import time
import random

class SharedBlackboard:
    """Stand-in for a Redis-backed shared memory store."""

    def __init__(self):
        self.state = {}
        self.locks = {}
        self.master_lock = threading.Lock()
        self.committed = []

    def lock_key(self, key, agent_name, timeout=2.0):
        start = time.time()
        while time.time() - start < timeout:
            with self.master_lock:
                if key not in self.locks:
                    self.locks[key] = agent_name
                    return True
            time.sleep(0.05)
        return False   # timed out -> avoids deadlock by giving up instead of blocking forever

    def unlock_key(self, key, agent_name):
        with self.master_lock:
            if self.locks.get(key) == agent_name:
                del self.locks[key]

    def write(self, key, value):
        self.state[key] = value

    def read(self, key):
        return self.state.get(key)

    def commit(self, record):
        # stand-in for a transactional DB INSERT/COMMIT
        with self.master_lock:
            self.committed.append(record)


class Agent:
    def __init__(self, name, role, blackboard):
        self.name = name
        self.role = role
        self.blackboard = blackboard

    def act(self, task_key):
        if not self.blackboard.lock_key(task_key, self.name):
            print(f"[{self.name}] could not acquire lock on {task_key}, backing off")
            return

        try:
            time.sleep(random.uniform(0.05, 0.2))  # simulate work (e.g. an LLM call)
            current = self.blackboard.read(task_key) or ""
            updated = current + f" | {self.role}:{self.name} processed"
            self.blackboard.write(task_key, updated)
            print(f"[{self.name}] updated '{task_key}' -> {updated}")
        finally:
            self.blackboard.unlock_key(task_key, self.name)

    def finalize(self, task_key):
        result = self.blackboard.read(task_key)
        self.blackboard.commit({"agent": self.name, "task": task_key, "result": result})


if __name__ == "__main__":
    board = SharedBlackboard()
    agents = [
        Agent("CodeGen-1", "CodeGenerator", board),
        Agent("Auditor-1", "SystemAuditor", board),
        Agent("QA-1", "QAAnalyst", board),
    ]

    task_key = "feature_x_pipeline"
    board.write(task_key, "start")

    threads = [threading.Thread(target=a.act, args=(task_key,)) for a in agents]
    for t in threads:
        t.start()
    for t in threads:
        t.join()

    agents[-1].finalize(task_key)
    print("\nCommitted records:", board.committed)

---
## Task 8: Multi-Agent Collaborative Swarm with Shared Blackboard